In [1]:
# ============================================
# Cell 1: Imports and directory setup
# ============================================

from pathlib import Path

import numpy as np
import pandas as pd

# Base project directory (where this notebook lives)
PROJECT_DIR = Path.cwd()

# Data directories
RAW_DIR = PROJECT_DIR / "raw"
PROCESSED_DIR = PROJECT_DIR / "processed"
BLS_DIR = RAW_DIR / "bls_food_at_home"

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("BLS food-at-home directory:", BLS_DIR)

# Quick existence check
print("\nDirectory existence check:")
print("RAW_DIR exists:", RAW_DIR.exists())
print("PROCESSED_DIR exists:", PROCESSED_DIR.exists())
print("BLS_DIR exists:", BLS_DIR.exists())


Project directory: /Users/arnavjain/DS_6021_Final_Project
Raw data directory: /Users/arnavjain/DS_6021_Final_Project/raw
Processed data directory: /Users/arnavjain/DS_6021_Final_Project/processed
BLS food-at-home directory: /Users/arnavjain/DS_6021_Final_Project/raw/bls_food_at_home

Directory existence check:
RAW_DIR exists: True
PROCESSED_DIR exists: True
BLS_DIR exists: True


In [4]:
# ============================================
# Cell 2 (final): Load raw CSV data robustly
# ============================================

from pandas.errors import ParserError

data_paths = {
    "acs_race": RAW_DIR / "ACSDT1Y2024.B02001-Data.csv",      # race breakdown
    "acs_income": RAW_DIR / "ACSDT1Y2024.B19013-Data.csv",    # median household income (if needed)
    "acs_poverty": RAW_DIR / "ACSST1Y2024.S1701-Data.csv",    # poverty status
    "rppi": RAW_DIR / "rpi_rpce_rpps.csv",                    # regional price parities / cost of living
    "rural_urban": RAW_DIR / "Ruralurbancontinuumcodes2023.csv",
    "snap_benefits": RAW_DIR / "snap-benefits-8.csv",
    "snap_persons": RAW_DIR / "snap-persons-8.csv",
    "state_trifecta": RAW_DIR / "state_trifecta_2024.csv",
    "snap_regions": RAW_DIR / "usda_snap_regions.csv",
}

dfs = {}

for name, path in data_paths.items():
    try:
        # Try standard CSV with forgiving decoding
        df = pd.read_csv(path, dtype=str, encoding_errors="replace")
        sep_used = ","
    except ParserError:
        # Fallback: try tab-separated (still forgiving decoding)
        try:
            df = pd.read_csv(path, dtype=str, sep="\t", encoding_errors="replace")
            sep_used = "\\t (tab)"
        except Exception as e:
            print(f"!! {name:15s} FAILED to load from {path.name}: {e}")
            continue
    except FileNotFoundError:
        print(f"!! {name:15s} NOT FOUND at {path}")
        continue

    dfs[name] = df
    print(f"{name:15s} loaded from {path.name:35s} sep={sep_used} -> shape {df.shape}")

# Unpack into named variables for convenience
acs_race       = dfs.get("acs_race")
acs_income     = dfs.get("acs_income")
acs_poverty    = dfs.get("acs_poverty")
rppi           = dfs.get("rppi")
rural_urban    = dfs.get("rural_urban")
snap_benefits  = dfs.get("snap_benefits")
snap_persons   = dfs.get("snap_persons")
state_trifecta = dfs.get("state_trifecta")
snap_regions   = dfs.get("snap_regions")


acs_race        loaded from ACSDT1Y2024.B02001-Data.csv         sep=, -> shape (53, 23)
acs_income      loaded from ACSDT1Y2024.B19013-Data.csv         sep=, -> shape (53, 5)
acs_poverty     loaded from ACSST1Y2024.S1701-Data.csv          sep=, -> shape (53, 375)
rppi            loaded from rpi_rpce_rpps.csv                   sep=\t (tab) -> shape (634, 1)
rural_urban     loaded from Ruralurbancontinuumcodes2023.csv    sep=, -> shape (9703, 5)
snap_benefits   loaded from snap-benefits-8.csv                 sep=, -> shape (66, 6)
snap_persons    loaded from snap-persons-8.csv                  sep=, -> shape (67, 6)
state_trifecta  loaded from state_trifecta_2024.csv             sep=, -> shape (50, 2)
snap_regions    loaded from usda_snap_regions.csv               sep=, -> shape (55, 2)


In [5]:
# ============================================
# Cell 3: Inspect schemas and sample rows
# ============================================

def show_info(name, df, n_cols=8, n_rows=5):
    if df is None:
        print(f"\n{name}: DataFrame is None")
        return
    print("\n" + "="*80)
    print(f"{name} — shape: {df.shape}")
    print("- Columns:")
    print(list(df.columns))
    print("- Sample:")
    # Show only the first few columns so it isn't overwhelming
    display(df.iloc[:n_rows, :n_cols])

# Inspect each key dataset
show_info("acs_race", acs_race)
show_info("acs_income", acs_income)
show_info("acs_poverty", acs_poverty)
show_info("rppi (cost of living)", rppi)
show_info("rural_urban", rural_urban)
show_info("snap_benefits", snap_benefits)
show_info("snap_persons", snap_persons)
show_info("state_trifecta", state_trifecta)
show_info("snap_regions", snap_regions)



acs_race — shape: (53, 23)
- Columns:
['GEO_ID', 'NAME', 'B02001_001E', 'B02001_001M', 'B02001_002E', 'B02001_002M', 'B02001_003E', 'B02001_003M', 'B02001_004E', 'B02001_004M', 'B02001_005E', 'B02001_005M', 'B02001_006E', 'B02001_006M', 'B02001_007E', 'B02001_007M', 'B02001_008E', 'B02001_008M', 'B02001_009E', 'B02001_009M', 'B02001_010E', 'B02001_010M', 'Unnamed: 22']
- Sample:


,GEO_ID,NAME,B02001_001E,B02001_001M,B02001_002E,B02001_002M,B02001_003E,B02001_003M
0,Geography,Geographic Area Name,Estimate!!Total:,Margin of Error!!Total:,Estimate!!Total:!!White alone,Margin of Error!!Total:!!White alone,Estimate!!Total:!!Black or African American alone,Margin of Error!!Total:!!Black or African Amer...
1,0400000US01,Alabama,5157699,*****,3301529,10356,1302107,9734
2,0400000US02,Alaska,740133,*****,434520,3307,18548,3228
3,0400000US04,Arizona,7582384,*****,4345673,23871,360868,13306
4,0400000US05,Arkansas,3088354,*****,2105944,10689,437805,7124



acs_income — shape: (53, 5)
- Columns:
['GEO_ID', 'NAME', 'B19013_001E', 'B19013_001M', 'Unnamed: 4']
- Sample:


,GEO_ID,NAME,B19013_001E,B19013_001M,Unnamed: 4
0,Geography,Geographic Area Name,Estimate!!Median household income in the past ...,Margin of Error!!Median household income in th...,NaN
1,0400000US01,Alabama,66659,780,NaN
2,0400000US02,Alaska,95665,3278,NaN
3,0400000US04,Arizona,81486,684,NaN
4,0400000US05,Arkansas,62106,831,NaN



acs_poverty — shape: (53, 375)
- Columns:
['GEO_ID', 'NAME', 'S1701_C01_001E', 'S1701_C01_001M', 'S1701_C01_002E', 'S1701_C01_002M', 'S1701_C01_003E', 'S1701_C01_003M', 'S1701_C01_004E', 'S1701_C01_004M', 'S1701_C01_005E', 'S1701_C01_005M', 'S1701_C01_006E', 'S1701_C01_006M', 'S1701_C01_007E', 'S1701_C01_007M', 'S1701_C01_008E', 'S1701_C01_008M', 'S1701_C01_009E', 'S1701_C01_009M', 'S1701_C01_010E', 'S1701_C01_010M', 'S1701_C01_011E', 'S1701_C01_011M', 'S1701_C01_012E', 'S1701_C01_012M', 'S1701_C01_013E', 'S1701_C01_013M', 'S1701_C01_014E', 'S1701_C01_014M', 'S1701_C01_015E', 'S1701_C01_015M', 'S1701_C01_016E', 'S1701_C01_016M', 'S1701_C01_017E', 'S1701_C01_017M', 'S1701_C01_018E', 'S1701_C01_018M', 'S1701_C01_019E', 'S1701_C01_019M', 'S1701_C01_020E', 'S1701_C01_020M', 'S1701_C01_021E', 'S1701_C01_021M', 'S1701_C01_022E', 'S1701_C01_022M', 'S1701_C01_023E', 'S1701_C01_023M', 'S1701_C01_024E', 'S1701_C01_024M', 'S1701_C01_025E', 'S1701_C01_025M', 'S1701_C01_026E', 'S1701_C01_026M', 'S

...

,SUPPLEMENTAL NUTRITION ASSISTANCE PROGRAM: NUMBER OF PERSONS PARTICIPATING,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,"(Data as of August 8, 2025)",NaN,NaN,NaN,NaN,NaN
1,State / Territory,May 2024\n,April 2025\nPreliminary,May 2025\nInitial,Percent Change\nMay 2025 vs\nApril 2025,Percent Change\nMay 2025 vs\nMay 2024
2,Alabama,"749,707","737,704","736,178",-0.2%,-1.8%
3,Alaska,"79,542","63,507","66,377",4.5%,-16.6%
4,American Samoa,--,--,--,--,--



state_trifecta — shape: (50, 2)
- Columns:
['state', 'trifecta_2024']
- Sample:


,state,trifecta_2024
0,Alabama,Republican
1,Alaska,Republican
2,Arizona,Divided
3,Arkansas,Republican
4,California,Democratic



snap_regions — shape: (55, 2)
- Columns:
['state', 'usda_snap_region']
- Sample:


,state,usda_snap_region
0,Delaware,Mid-Atlantic
1,District of Columbia,Mid-Atlantic
2,Maryland,Mid-Atlantic
3,New Jersey,Mid-Atlantic
4,Pennsylvania,Mid-Atlantic
